# 03 · ASR: numbers become words

Whisper (`whisper-large-v3-turbo`), served by [Groq](https://groq.com).

- **Encoder**: reads the whole clip once
- **Decoder**: writes one token at a time

## 3.1 · Setup

API key in `.env` (gitignored), never in a cell.

In [1]:
import io
import os
import statistics
import time
from pathlib import Path

import numpy as np
import soundfile as sf
from dotenv import load_dotenv
from groq import Groq

load_dotenv(Path("../.env"))
client = Groq(api_key=os.environ["GROQ_API_KEY"])

SAMPLE_RATE = 16_000
audio, sr = sf.read("../recordings/sample.wav", dtype="float32")
assert sr == SAMPLE_RATE

print(f"audio  : {len(audio)/sr:.1f}s @ {sr} Hz")
print(f"api key: {'loaded' if os.environ.get('GROQ_API_KEY') else 'MISSING'}")

audio  : 10.0s @ 16000 Hz
api key: loaded


## 3.2 · Send audio, get text

The numbers are packed as a WAV file and uploaded.

In [2]:
def transcribe(samples: np.ndarray, model: str = "whisper-large-v3-turbo") -> tuple[float, str]:
    """Send audio to Groq. Returns (latency_ms, text)."""
    buf = io.BytesIO()
    sf.write(buf, samples, SAMPLE_RATE, format="WAV")   # serialize in memory
    buf.seek(0)

    t0 = time.perf_counter()
    result = client.audio.transcriptions.create(
        file=("audio.wav", buf.read()),
        model=model,
        response_format="json",
    )
    return (time.perf_counter() - t0) * 1000, result.text


latency, text = transcribe(audio)
print(f"{latency:.0f} ms")
print(repr(text))

441 ms
" So I'm building a voice dictation app and I want it to actually work."


Capitals and punctuation came for free. Fillers survive: that is `04`.

## 3.3 · One measurement is not a measurement

Five calls per model: min, median, max.

In [3]:
def benchmark(samples: np.ndarray, model: str, runs: int = 5) -> dict:
    latencies, text = [], ""
    for _ in range(runs):
        ms, text = transcribe(samples, model)
        latencies.append(ms)
    latencies.sort()
    return {
        "model": model,
        "min": latencies[0],
        "median": statistics.median(latencies),
        "max": latencies[-1],
        "spread": latencies[-1] - latencies[0],
        "text": text,
    }


results = []
for model in ["whisper-large-v3-turbo", "whisper-large-v3"]:
    r = benchmark(audio, model)
    results.append(r)
    print(f"{r['model']:24s} min={r['min']:6.0f}  median={r['median']:6.0f}  "
          f"max={r['max']:6.0f}  spread={r['spread']:5.0f} ms")
    print(f"{'':24s} -> {r['text']!r}\n")

whisper-large-v3-turbo   min=   316  median=   461  max=   486  spread=  170 ms
                         -> " So I'm building a voice dictation app and I want it to actually work."

whisper-large-v3         min=   333  median=   383  max=   659  spread=  326 ms
                         -> " So I'm building a voice dictation app and I want it to actually work."



## 3.4 · Whisper hallucinates on silence

Three seconds of pure zeros. The correct answer is empty text.

In [4]:
silence = np.zeros(3 * SAMPLE_RATE, dtype="float32")

for i in range(3):
    ms, text = transcribe(silence)
    print(f"run {i+1}: {ms:5.0f} ms -> {text!r}")

run 1:   474 ms -> ' Thank you.'
run 2:   279 ms -> ' Thank you.'
run 3:   435 ms -> ' Thank you.'


The decoder has no way to say nothing.

## 3.5 · Trim the silence first

Send only what the VAD calls speech, and time both.

In [6]:
import torch
from silero_vad import load_silero_vad, get_speech_timestamps

vad = load_silero_vad(onnx=True)
vad.reset_states()
segments = get_speech_timestamps(
    audio, vad,
    sampling_rate=SAMPLE_RATE,
    min_silence_duration_ms=500,   # the "balanced" setting from notebook 02
    speech_pad_ms=100,
)
speech_only = np.concatenate([audio[s["start"]:s["end"]] for s in segments])

saved = 100 * (1 - len(speech_only) / len(audio))
print(f"{len(segments)} segments, {len(speech_only)/SAMPLE_RATE:.2f}s kept "
      f"of {len(audio)/SAMPLE_RATE:.2f}s  ({saved:.0f}% less audio)\n")

full = results[0]                                        # turbo on the full clip
trimmed = benchmark(speech_only, "whisper-large-v3-turbo")

print(f"{'full 10s':14s} min={full['min']:6.0f}  median={full['median']:6.0f}  max={full['max']:6.0f} ms")
print(f"{'VAD-trimmed':14s} min={trimmed['min']:6.0f}  median={trimmed['median']:6.0f}  max={trimmed['max']:6.0f} ms")
print()
print(f"median improved by {full['median'] - trimmed['median']:.0f} ms")
print(f"worst case improved by {full['max'] - trimmed['max']:.0f} ms")
print()
print(f"full    -> {full['text']!r}")
print(f"trimmed -> {trimmed['text']!r}")

2 segments, 5.84s kept of 10.00s  (42% less audio)

full 10s       min=   316  median=   461  max=   486 ms
VAD-trimmed    min=   278  median=   296  max=   361 ms

median improved by 165 ms
worst case improved by 125 ms

full    -> " So I'm building a voice dictation app and I want it to actually work."
trimmed -> " So I'm building a voice dictation app and I wanted to actually work"


## 3.6 · Against the budget

In [7]:
VAD_MS = 71          # measured in notebook 02
BUDGET = {"capture": None, "VAD": 200, "ASR": 200, "LLM cleanup": 200, "inject": None}

rows = [
    ("2. capture",        "~0 ms",                        "included below"),
    ("3. VAD",            f"{VAD_MS} ms",                 "~200 ms (shared)"),
    ("4. ASR  median",    f"{trimmed['median']:.0f} ms",  "~200 ms"),
    ("4. ASR  worst",     f"{trimmed['max']:.0f} ms",     "~200 ms"),
    ("5. LLM cleanup",    "not built yet",                "~200 ms"),
    ("6. inject",         "not built yet",                "in the slack"),
    ("   network",        "in ASR above",                 "~200 ms"),
]

print(f"{'stage':<20}{'ours':>16}{'Wispr budget':>20}")
print("-" * 56)
for stage, ours, budget in rows:
    print(f"{stage:<20}{ours:>16}{budget:>20}")
print("-" * 56)

median_total = VAD_MS + trimmed["median"]
worst_total = VAD_MS + trimmed["max"]
print(f"{'TOTAL so far':<20}{f'{median_total:.0f} ms':>16}{'700 ms p99':>20}")
print(f"{'  worst case':<20}{f'{worst_total:.0f} ms':>16}{'':>20}")
print()
print(f"Headroom left for the LLM pass: {700 - worst_total:.0f} ms at our worst case.")
print("Note: our 'network' is already inside the ASR number, because Groq is a")
print("remote call. Wispr's 200ms ASR budget is compute only. Not comparable yet.")

stage                           ours        Wispr budget
--------------------------------------------------------
2. capture                     ~0 ms      included below
3. VAD                         71 ms    ~200 ms (shared)
4. ASR  median                296 ms             ~200 ms
4. ASR  worst                 361 ms             ~200 ms
5. LLM cleanup         not built yet             ~200 ms
6. inject              not built yet        in the slack
   network              in ASR above             ~200 ms
--------------------------------------------------------
TOTAL so far                  367 ms          700 ms p99
  worst case                  432 ms                    

Headroom left for the LLM pass: 268 ms at our worst case.
Note: our 'network' is already inside the ASR number, because Groq is a
remote call. Wispr's 200ms ASR budget is compute only. Not comparable yet.
